# AI-Powered Resume Screening & Job Matching System

## Notebook 2: Transformer-based Resume-Job Matching

### Objective
Build a Transformer model that predicts whether a resume matches a job description.

In [1]:
!pip install faiss-cpu

In [2]:
import os
import random
import warnings

import faiss
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

In [3]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google'

In [ ]:
BASE_PATH = "/content/drive/MyDrive/AI Resume Screening/"
DATA_PATH = BASE_PATH + "data/"
MODEL_PATH = BASE_PATH + "models/"
OUTPUT_PATH = BASE_PATH + "output/"

os.makedirs(DATA_PATH, exist_ok=True)
os.makedirs(MODEL_PATH, exist_ok=True)
os.makedirs(OUTPUT_PATH, exist_ok=True)

print("Data :", os.path.exists(DATA_PATH))
print("Models :", os.path.exists(MODEL_PATH))
print("Output :", os.path.exists(OUTPUT_PATH))

Data : True
Models : True
Output : True


In [ ]:
resume_df = pd.read_csv(DATA_PATH + "processed_resume.csv")
job_df2 = pd.read_csv(DATA_PATH + "processed_jobs.csv")

resume_embeddings = np.load(MODEL_PATH + "resume_embeddings.npy")

index = faiss.read_index(MODEL_PATH + "job_faiss.index")

print("FAISS Index Size:", index.ntotal)

FAISS Index Size: 1615940


In [ ]:
print(np.linalg.norm(resume_embeddings[0]))
# print(np.linalg.norm(job_embeddings[0]))

0.99999994


In [ ]:
print("="*60)

print("Resume Shape :", resume_df.shape)

print("Job Shape :", job_df2.shape)

print("Resume Embeddings :", resume_embeddings.shape)

# print("Job Embeddings :", job_embeddings.shape)

print("="*60)

Resume Shape : (2484, 6)
Job Shape : (1615940, 15)
Resume Embeddings : (2484, 384)


In [ ]:
resume_embeddings = resume_embeddings.astype(np.float32)
# job_embeddings = job_embeddings.astype(np.float32)

In [ ]:
print("FAISS Index Size :", index.ntotal)

FAISS Index Size : 1615940


In [ ]:
resume_idx = 0

D, I = index.search(
    resume_embeddings[resume_idx].reshape(1, -1),
    k=10
)

results = job_df2.iloc[I[0]][[
    "Job Title",
    "Role",
    "Company"
]].copy()

results["Similarity"] = D[0].round(4)

results

,Job Title,Role,Company,Similarity
1320547,Data Engineer,ETL Developer,Olin,0.6806
1247505,Executive Assistant,Personal Assistant,Danaher,0.6806
874038,Executive Assistant,Personal Assistant,Pfizer Inc.,0.6806
771303,Software Engineer,Backend Developer,McDonald's,0.6806
753513,Veterinarian,Equine Veterinarian,Agricultural Bank of China,0.6806
637384,Substance Abuse Counselor,Family Counselor,Legal & General Group,0.6806
584463,Nurse Practitioner,Family Nurse Practitioner,XPO,0.6806
1074019,Brand Manager,Brand Marketing Analyst,"Wirecard AG (Now defunct, due to bankruptcy)",0.6804
756015,Project Manager,IT Project Manager,New York Life Insurance,0.6804
385068,Content Writer,Content Strategist,Prudential plc,0.6804


In [ ]:
print("Top Job Index :", I[0][0])
print("Similarity :", D[0][0])

Top Job Index : 1320547
Similarity : 0.68060446


In [ ]:
from tqdm.auto import tqdm

In [ ]:
from tqdm.auto import tqdm
import numpy as np

pairs = []

TOP_K = 20          # Retrieve top 20 jobs
POSITIVE_K = 5      # Top 5 as positive
NEGATIVE_K = 3      # Last 3 as hard negative
EASY_NEGATIVE_K = 2

num_jobs = len(job_df2)

for resume_idx in tqdm(range(len(resume_df)), desc="Generating Resume-Job Pairs"):

    # --------------------------------------------------
    # Search Top-K Jobs
    # --------------------------------------------------
    D, I = index.search(
        resume_embeddings[resume_idx].reshape(1, -1),
        k=TOP_K
    )

    retrieved_jobs = I[0]
    scores = D[0]

    resume_text = resume_df.iloc[resume_idx]["resume_text"]

    # ==================================================
    # Positive Pairs (Top-5)
    # ==================================================
    for job_idx, score in zip(
        retrieved_jobs[:POSITIVE_K],
        scores[:POSITIVE_K]
    ):

        pairs.append({

            "resume_idx": resume_idx,
            "job_idx": int(job_idx),

            "resume_text": resume_text,
            "job_text": job_df2.iloc[job_idx]["job_text"],

            "Job Title": job_df2.iloc[job_idx]["Job Title"],
            "Role": job_df2.iloc[job_idx]["Role"],
            "Company": job_df2.iloc[job_idx]["Company"],

            "similarity": float(score),

            "label": 1

        })

    # ==================================================
    # Hard Negative Pairs
    # ==================================================
    for job_idx, score in zip(
        retrieved_jobs[-NEGATIVE_K:],
        scores[-NEGATIVE_K:]
    ):

        pairs.append({

            "resume_idx": resume_idx,
            "job_idx": int(job_idx),

            "resume_text": resume_text,
            "job_text": job_df2.iloc[job_idx]["job_text"],

            "Job Title": job_df2.iloc[job_idx]["Job Title"],
            "Role": job_df2.iloc[job_idx]["Role"],
            "Company": job_df2.iloc[job_idx]["Company"],

            "similarity": float(score),

            "label": 0

        })

    # ==================================================
    # Easy Negative Pairs
    # ==================================================
    random_jobs = np.random.choice(
        num_jobs,
        EASY_NEGATIVE_K,
        replace=False
    )

    for job_idx in random_jobs:

        # Skip if the random job is already in Top-K
        if job_idx in retrieved_jobs:
            continue

        pairs.append({

            "resume_idx": resume_idx,
            "job_idx": int(job_idx),

            "resume_text": resume_text,
            "job_text": job_df2.iloc[job_idx]["job_text"],

            "Job Title": job_df2.iloc[job_idx]["Job Title"],
            "Role": job_df2.iloc[job_idx]["Role"],
            "Company": job_df2.iloc[job_idx]["Company"],

            "similarity": 0.0,

            "label": 0

        })

print("=" * 60)
print("Pair Generation Completed")
print("=" * 60)
print("Total Resume-Job Pairs:", len(pairs))

Generating Resume-Job Pairs:   0%|          | 0/2484 [00:00<?, ?it/s]

Pair Generation Completed
Total Resume-Job Pairs: 24840


In [ ]:
pairs_df = pd.DataFrame(pairs)

pairs_df.drop_duplicates(
    subset=["resume_idx", "job_idx"],
    inplace=True
)

pairs_df.reset_index(drop=True, inplace=True)

print("Dataset Shape :", pairs_df.shape)
print("\nLabel Distribution:\n")
print(pairs_df["label"].value_counts())

pairs_df.head()

Dataset Shape : (24840, 9)

Label Distribution:

label
1    12420
0    12420
Name: count, dtype: int64


,resume_idx,job_idx,resume_text,job_text,Job Title,Role,Company,similarity,label
0,0,1320547,hr administrator/marketing associate hr admini...,data engineer etl developer an etl developer s...,Data Engineer,ETL Developer,Olin,0.680604,1
1,0,1247505,hr administrator/marketing associate hr admini...,executive assistant personal assistant a perso...,Executive Assistant,Personal Assistant,Danaher,0.680604,1
2,0,874038,hr administrator/marketing associate hr admini...,executive assistant personal assistant a perso...,Executive Assistant,Personal Assistant,Pfizer Inc.,0.680604,1
3,0,771303,hr administrator/marketing associate hr admini...,software engineer backend developer a backend ...,Software Engineer,Backend Developer,McDonald's,0.680604,1
4,0,753513,hr administrator/marketing associate hr admini...,veterinarian equine veterinarian an equine vet...,Veterinarian,Equine Veterinarian,Agricultural Bank of China,0.680604,1


In [ ]:
print("\nSimilarity Statistics")
print(pairs_df["similarity"].describe())


Similarity Statistics
count    24840.000000
mean         0.567469
std          0.286820
min          0.000000
25%          0.637752
50%          0.693396
75%          0.731591
max          0.880753
Name: similarity, dtype: float64


In [ ]:
pairs_df.sample(10)

,resume_idx,job_idx,resume_text,job_text,Job Title,Role,Company,similarity,label
12727,1272,423652,senior digital producer/multimedia specialist ...,civil engineer water resources engineer a wate...,Civil Engineer,Water Resources Engineer,Raytheon Technologies,0.682401,0
4033,403,753943,substitute teacher summary health administrati...,interior designer hospitality interior designe...,Interior Designer,Hospitality Interior Designer,CNOOC (China National Offshore Oil Corporation),0.619501,1
11594,1159,413551,consultant executive profile as a professional...,ux/ui designer user experience designer user e...,UX/UI Designer,User Experience Designer,CarMax,0.651711,1
13079,1307,1332428,digital strategy manager professional profile ...,seo analyst seo content strategist as an seo c...,SEO Analyst,SEO Content Strategist,Adelaide Brighton,0.000000,0
7330,733,1567450,recruiter tax exempt markets professional summ...,structural engineer bridge engineer a bridge e...,Structural Engineer,Bridge Engineer,Robert Half International,0.678250,1
10142,1014,1263405,sales summary to obtain a position where i can...,project manager construction project manager t...,Project Manager,Construction Project Manager,Salesforce,0.757865,1
22656,2265,1463014,arts instructor summary motivated teaching pro...,registered nurse charge nurse a charge nurse l...,Registered Nurse,Charge Nurse,Olin,0.766750,0
10976,1097,563897,sales associate summary to further my career i...,software engineer frontend developer a fronten...,Software Engineer,Frontend Developer,Kellogg,0.736406,0
12384,1238,1374955,director it client services films operations a...,software architect solution architect a soluti...,Software Architect,Solution Architect,"Wyndham Hotels & Resorts, Inc.",0.662377,1
7582,758,1343247,hiring manager/recruiting manager summary self...,product manager agile product owner agile prod...,Product Manager,Agile Product Owner,Henkel AG & Co. KGaA,0.748857,1


In [ ]:
print("Total Training Pairs :", len(pairs_df))
print(pairs_df["label"].value_counts())

Total Training Pairs : 24840
label
1    12420
0    12420
Name: count, dtype: int64


In [ ]:
pairs_df["label"].value_counts()

,count
label,
1,12420
0,12420


In [ ]:
pairs_df = pairs_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

In [ ]:
pairs_df.drop_duplicates(
    subset=["resume_idx", "job_idx"],
    inplace=True
)

pairs_df.reset_index(drop=True, inplace=True)

In [ ]:
print("Total Pairs :", len(pairs_df))
print("Positive :", (pairs_df["label"] == 1).sum())
print("Negative :", (pairs_df["label"] == 0).sum())

Total Pairs : 24840
Positive : 12420
Negative : 12420


In [ ]:
pairs_df.to_csv(
    OUTPUT_PATH + "resume_job_pairs.csv",
    index=False
)
print(pairs_df.head())

print("\nShape:", pairs_df.shape)

print("Dataset Saved Successfully")

   resume_idx  job_idx                                        resume_text  \
0         516  1220095  sexual assault crisis counselor / victim advoc...   
1         456  1509288  benefit advocate career overview customer serv...   
2        1704  1518461  engineering intern summary automotive engineer...   
3        2084   175192  insurance specialist summary accomplished comm...   
4        1371   997183  main chef professional summary lead emergency ...   

                                            job_text  \
0  aerospace engineer avionics engineer avionics ...   
1  accountant tax accountant tax accountants spec...   
2  business development manager market expansion ...   
3  public relations specialist brand pr specialis...   
4  social media coordinator content creator creat...   

                      Job Title                      Role  \
0            Aerospace Engineer         Avionics Engineer   
1                    Accountant            Tax Accountant   
2  Business Devel

In [ ]:
print("=" * 50)
print("Dataset Statistics")
print("=" * 50)

print("Total Pairs :", len(pairs_df))
print("Positive    :", (pairs_df["label"] == 1).sum())
print("Negative    :", (pairs_df["label"] == 0).sum())

print("\nSimilarity Statistics")
print(pairs_df["similarity"].describe())

In [ ]:
print(pairs_df["label"].value_counts())

print("\nShape:", pairs_df.shape)

label
1    12420
0    12420
Name: count, dtype: int64

Shape: (24840, 9)


# Transformer

In [ ]:
!pip install -q transformers datasets accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.6 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

import evaluate

In [ ]:
pairs_df = pd.read_csv(
    OUTPUT_PATH + "resume_job_pairs.csv"
)

pairs_df.head()

,resume_idx,job_idx,resume_text,job_text,Job Title,Role,Company,similarity,label
0,516,1220095,sexual assault crisis counselor / victim advoc...,aerospace engineer avionics engineer avionics ...,Aerospace Engineer,Avionics Engineer,Knight-Swift Transportation Holdings,0.672243,1
1,456,1509288,benefit advocate career overview customer serv...,accountant tax accountant tax accountants spec...,Accountant,Tax Accountant,Otis Worldwide,0.000000,0
2,1704,1518461,engineering intern summary automotive engineer...,business development manager market expansion ...,Business Development Manager,Market Expansion Manager,Alphabet,0.688631,1
3,2084,175192,insurance specialist summary accomplished comm...,public relations specialist brand pr specialis...,Public Relations Specialist,Brand PR Specialist,Live Nation Entertainment,0.700707,1
4,1371,997183,main chef professional summary lead emergency ...,social media coordinator content creator creat...,Social Media Coordinator,Content Creator,Metro Bank,0.637076,1


In [ ]:
pairs_df = pairs_df[
    [
        "resume_text",
        "job_text",
        "label"
    ]
]

pairs_df.head()

,resume_text,job_text,label
0,sexual assault crisis counselor / victim advoc...,aerospace engineer avionics engineer avionics ...,1
1,benefit advocate career overview customer serv...,accountant tax accountant tax accountants spec...,0
2,engineering intern summary automotive engineer...,business development manager market expansion ...,1
3,insurance specialist summary accomplished comm...,public relations specialist brand pr specialis...,1
4,main chef professional summary lead emergency ...,social media coordinator content creator creat...,1


In [ ]:
train_df, val_df = train_test_split(

    pairs_df,

    test_size=0.2,

    random_state=42,

    stratify=pairs_df["label"]

)

print("Training Samples :", len(train_df))
print("Validation Samples :", len(val_df))

Training Samples : 19872
Validation Samples : 4968


In [ ]:
train_dataset = Dataset.from_pandas(
    train_df.reset_index(drop=True)
)

val_dataset = Dataset.from_pandas(
    val_df.reset_index(drop=True)
)

print(train_dataset)
print(val_dataset)

Dataset({
    features: ['resume_text', 'job_text', 'label'],
    num_rows: 19872
})
Dataset({
    features: ['resume_text', 'job_text', 'label'],
    num_rows: 4968
})


In [ ]:
MODEL_NAME = "microsoft/MiniLM-L12-H384-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
print(train_df.isnull().sum())

print(val_df.isnull().sum())

resume_text    8
job_text       0
label          0
dtype: int64
resume_text    2
job_text       0
label          0
dtype: int64


In [ ]:
print(train_df.dtypes)

resume_text    object
job_text       object
label           int64
dtype: object


In [ ]:
train_df["resume_text"] = train_df["resume_text"].fillna("").astype(str)
train_df["job_text"] = train_df["job_text"].fillna("").astype(str)

val_df["resume_text"] = val_df["resume_text"].fillna("").astype(str)
val_df["job_text"] = val_df["job_text"].fillna("").astype(str)

In [ ]:
train_dataset = Dataset.from_pandas(
    train_df.reset_index(drop=True)
)

val_dataset = Dataset.from_pandas(
    val_df.reset_index(drop=True)
)

In [ ]:
MAX_LENGTH = 384

def tokenize_function(batch):

    resume_texts = [str(x) for x in batch["resume_text"]]
    job_texts = [str(x) for x in batch["job_text"]]

    return tokenizer(
        resume_texts,
        job_texts,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

In [ ]:
train_dataset = train_dataset.map(
    tokenize_function,
    batched=True
)

val_dataset = val_dataset.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/19872 [00:00<?, ? examples/s]

Map:   0%|          | 0/4968 [00:00<?, ? examples/s]

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

print(model.config)

pytorch_model.bin:   0%|          | 0.00/133M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: microsoft/MiniLM-L12-H384-uncased
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertConfig {
  "add_cross_attention": false,
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 384,
  "initializer_range": 0.02,
  "intermediate_size": 1536,
  "is_decoder": false,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "tie_word_embeddings": true,
  "transformers_version": "5.0.0",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}



In [ ]:
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy.compute(
            predictions=predictions,
            references=labels
        )["accuracy"],

        "precision": precision.compute(
            predictions=predictions,
            references=labels
        )["precision"],

        "recall": recall.compute(
            predictions=predictions,
            references=labels
        )["recall"],

        "f1": f1.compute(
            predictions=predictions,
            references=labels
        )["f1"]
    }

In [ ]:
training_args = TrainingArguments(

    output_dir="./resume_match_model",

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    num_train_epochs=3,

    weight_decay=0.01,

    load_best_model_at_end=True,

    metric_for_best_model="f1",

    greater_is_better=True,

    logging_steps=100,

    report_to="none",

    fp16=True,
    save_total_limit=2,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

In [ ]:
results = trainer.evaluate()

print(f"Accuracy : {results['eval_accuracy']:.4f}")
print(f"Precision: {results['eval_precision']:.4f}")
print(f"Recall   : {results['eval_recall']:.4f}")
print(f"F1 Score : {results['eval_f1']:.4f}")

In [ ]:
SAVE_PATH = MODEL_PATH + "resume_matching_model"

trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("Model saved successfully!")

In [ ]:
import torch

# Put model in evaluation mode
model.eval()

# Get the device where the model is
device = next(model.parameters()).device

sample = val_df.sample(1, random_state=42)

resume = sample.iloc[0]["resume_text"]
job = sample.iloc[0]["job_text"]

inputs = tokenizer(
    resume,
    job,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=256
)

# Move inputs to the same device as the model
inputs = {k: v.to(device) for k, v in inputs.items()}

# Disable gradient calculation
with torch.no_grad():
    outputs = model(**inputs)

probabilities = torch.softmax(outputs.logits, dim=1)

match_score = probabilities[0][1].item() * 100

print(f"Match Score: {match_score:.2f}%")

In [ ]:
import torch

model.eval()
device = next(model.parameters()).device

samples = val_df.sample(10, random_state=42)

for i in range(len(samples)):

    resume = samples.iloc[i]["resume_text"]
    job = samples.iloc[i]["job_text"]
    true_label = samples.iloc[i]["label"]

    inputs = tokenizer(
        resume,
        job,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probabilities = torch.softmax(outputs.logits, dim=1)

    match_score = probabilities[0][1].item() * 100

    print(
        f"Sample {i+1}: "
        f"True Label = {true_label}, "
        f"Match Score = {match_score:.2f}%"
    )